# Step 1 — Freeze the 2000–2018 best-paper cohort (v1)

Retain every in-range award record, including unmatched papers and authorship entries without author IDs. Do not choose a reference award per author yet.

## Scope and guarantees
- Source: `data/matched/huang_matched_openalex.csv`, the source used by the paper-workflow notebooks. Compare its bytes with `conf_paper/conf_data/huang_matched_openalex.csv` if available.
- Freeze the **observed award records in this source**, not all awards that may exist in the world. Eligibility here is year 2000–2018 inclusive; no new award-type or venue exclusions are invented.
- A valid-looking OpenAlex work ID means an existing recorded match, **not a newly validated match**.
- Preserve duplicates with flags; do not silently merge papers or author entries.
- Keep original author text when structured authorships are missing or malformed. Do not invent author identities by splitting unstructured names.
- Parse JSON/Python literal lists safely; never use `eval`, unpickle caches, call APIs, or change source files.
- The default run writes a new versioned directory once. It will not overwrite an existing freeze. Set `WRITE_FREEZE = False` for a preview only.

Suggested notebook location: `notebooks/01_freeze_award_cohort.ipynb`. It also works elsewhere inside the repository. The snapshot is created only when this notebook runs against your local data; this delivered notebook has no fabricated execution results.

In [1]:
from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone
import ast
import hashlib
import json
import re
import shutil
import subprocess
import tempfile
import unicodedata
import pandas as pd
from IPython.display import display

ROOT_OVERRIDE = None
SOURCE_REL = 'data/matched/huang_matched_openalex.csv'
COPY_REL = 'conf_paper/conf_data/huang_matched_openalex.csv'
VERSION = 'award_cohort_v1'
YEAR_START, YEAR_END = 2000, 2018
WRITE_FREEZE = True
ACK_SOURCE_COPY_DIFFERENCE = False
ACK_CHANGED_PAPER_COUNTS = False

# Historical paper summary, used as a warning/gate, never as generated results.
HISTORICAL_PAPER_ROWS = 925
HISTORICAL_NO_OPENALEX_ID = 22

def find_root():
    if ROOT_OVERRIDE:
        p = Path(ROOT_OVERRIDE).expanduser().resolve()
        if not (p / SOURCE_REL).is_file():
            raise FileNotFoundError('The configured root does not contain the award source.')
        return p
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (p / SOURCE_REL).is_file() and (p / 'conf_paper').is_dir():
            return p
    raise FileNotFoundError('Run inside the repository, or set ROOT_OVERRIDE.')

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def git_value(args):
    try:
        r = subprocess.run(['git', '-C', str(ROOT), *args], capture_output=True,
                           text=True, timeout=10)
        return r.stdout.strip() if r.returncode == 0 else None
    except (OSError, subprocess.TimeoutExpired):
        return None

ROOT = find_root()
TARGET = ROOT / 'conf_paper' / 'conf_data' / 'frozen' / VERSION
source_bytes = (ROOT / SOURCE_REL).read_bytes()
SOURCE_SHA256 = sha256_bytes(source_bytes)
copy_bytes = (ROOT / COPY_REL).read_bytes() if (ROOT / COPY_REL).exists() else None
COPY_SHA256 = sha256_bytes(copy_bytes) if copy_bytes is not None else None
SOURCE_COPY_DIFFER = COPY_SHA256 is not None and COPY_SHA256 != SOURCE_SHA256
source = pd.read_csv(BytesIO(source_bytes), dtype=str, keep_default_na=False, encoding='utf-8-sig')
required = {'year', 'conference', 'paper_title', 'openalex_id', 'authorships'}
if not required.issubset(source.columns):
    raise ValueError(f'Missing required source columns: {sorted(required - set(source.columns))}')
source.insert(0, '_source_row_number', range(1, len(source) + 1))
print('Source rows:', len(source))
print('Source SHA-256:', SOURCE_SHA256)
print('Journal copy:', 'missing' if copy_bytes is None else ('DIFFERENT — review before freezing' if SOURCE_COPY_DIFFER else 'byte-identical'))
print('Destination:', TARGET)

Source rows: 925
Source SHA-256: 566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421d16b75b389f5be4bdd
Journal copy: byte-identical
Destination: B:\Semester 4 UU\thesis-best-paper-trajectories\conf_paper\conf_data\frozen\award_cohort_v1


## 1. Define the cohort and retain source lineage

Rows outside the year range, and rows with invalid years, go into a separate exclusions file. They are never silently discarded from the archived source. Missing title/conference and duplicate keys are flagged but do not remove an in-range award record.

`cohort_record_id` identifies one source record within this frozen snapshot. It is derived from the complete source hash and source row number; it is not a universal paper identifier. All original fields are retained under `source_...` columns.

In [2]:
def blank(value):
    return value is None or str(value).strip().casefold() in {'', 'nan', 'none', 'null', '<na>'}

def oa_short(value, prefix):
    if blank(value):
        return None
    text = str(value).strip()
    match = re.fullmatch(r'(?:https?://openalex\.org/)?(' + prefix + r'\d+)/?', text, flags=re.I)
    return match.group(1).upper() if match else None

def text_key(value):
    return ' '.join(unicodedata.normalize('NFKC', str(value)).casefold().split())

years = pd.to_numeric(source.year, errors='coerce')
valid_year = years.notna() & years.mod(1).eq(0)
in_range = valid_year & years.between(YEAR_START, YEAR_END)
excluded = source.loc[~in_range].copy()
excluded['exclusion_reason'] = ['year_outside_2000_2018' if valid_year.loc[i] else 'invalid_or_missing_year' for i in excluded.index]
selected = source.loc[in_range].copy().reset_index(drop=True)
if selected.empty:
    raise ValueError('No in-range award records; do not freeze an empty cohort.')

papers = selected.rename(columns={c: 'source_' + c for c in selected.columns if c != '_source_row_number'}).copy()
papers = papers.rename(columns={'_source_row_number': 'source_row_number'})
papers.insert(0, 'cohort_record_id', [f'{SOURCE_SHA256}:row:{n}' for n in papers.source_row_number])
papers['award_year'] = pd.to_numeric(selected.year).astype(int)
papers['conference'] = selected.conference.str.strip()
papers['paper_title'] = selected.paper_title.str.strip()
papers['openalex_work_id'] = selected.openalex_id.map(lambda x: oa_short(x, 'W'))
papers['paper_missing_openalex_id'] = selected.openalex_id.map(blank)
papers['paper_invalid_openalex_id'] = ~papers.paper_missing_openalex_id & papers.openalex_work_id.isna()
papers['paper_has_recorded_openalex_match'] = papers.openalex_work_id.notna()
papers['paper_missing_conference'] = selected.conference.map(blank)
papers['paper_missing_title'] = selected.paper_title.map(blank)
papers['conference_key'] = selected.conference.map(text_key)
papers['title_key'] = selected.paper_title.map(text_key)
papers['duplicate_conference_year_title'] = papers.duplicated(['conference_key', 'award_year', 'title_key'], keep=False)
papers['duplicate_openalex_work_id'] = papers.openalex_work_id.notna() & papers.openalex_work_id.duplicated(keep=False)
original_columns = [c for c in selected if c != '_source_row_number']
papers['duplicate_exact_source_record'] = selected.duplicated(original_columns, keep=False)
assert len(papers) + len(excluded) == len(source)
assert papers.cohort_record_id.is_unique
assert papers.award_year.between(YEAR_START, YEAR_END).all()
display(papers[['cohort_record_id', 'conference', 'award_year', 'paper_title',
                'openalex_work_id', 'paper_missing_openalex_id', 'paper_invalid_openalex_id']].head())

,cohort_record_id,conference,award_year,paper_title,openalex_work_id,paper_missing_openalex_id,paper_invalid_openalex_id
0,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,AAAI,2018,Memory-Augmented Monte Carlo Tree Search,W2788603415,False,False
1,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,ACL,2018,Finding syntax in human encephalography with b...,W2798397965,False,False
2,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,ACL,2018,Learning to Ask Good Questions: Ranking Clarif...,W2963033005,False,False
3,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,ACL,2018,"Let's do it ""again"": A First Computational App...",W2899300759,False,False
4,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,CHI,2018,Agile 3D Sketching with Air Scaffolding,W4255717503,False,False


## 2. Expand authorships without dropping unidentified authors

A parsed authorship slot is retained even when its author ID is absent/invalid. Papers with no usable structured list receive one explicit **placeholder row** so they remain visible in joins. Placeholders are not counted as known authors or known authorship slots.

Raw author names, affiliations, and the original authorship entry are preserved where provided. `author_order_in_source` is the original list index, not independently verified PDF author order. Institutions are recorded from the paper metadata, not inferred from a current author profile.

In [3]:
def parse_authorships(value):
    if blank(value):
        return None, 'missing'
    try:
        parsed = json.loads(str(value))
    except (ValueError, TypeError):
        try:
            parsed = ast.literal_eval(str(value))
        except (ValueError, SyntaxError, TypeError, RecursionError):
            return None, 'parse_error'
    if not isinstance(parsed, list):
        return None, 'not_a_list'
    return parsed, 'empty_list' if not parsed else 'parsed'

AUTHOR_COLUMNS = ['cohort_record_id', 'authorship_record_id', 'conference', 'award_year',
    'openalex_work_id', 'author_order_in_source', 'is_placeholder', 'authorship_parse_status',
    'authorship_entry_invalid', 'author_id_raw', 'author_id', 'author_id_missing',
    'author_id_invalid', 'author_identity_unresolved', 'author_name', 'author_position',
    'institutions_json', 'raw_affiliation_strings_json', 'authorship_entry_json']
author_rows, paper_auth_flags = [], []
for p in papers.to_dict('records'):
    parsed, status = parse_authorships(p['source_authorships'])
    base = {c: p[c] for c in ['cohort_record_id', 'conference', 'award_year', 'openalex_work_id']}
    missing_count, invalid_count, valid_count = 0, 0, 0
    if parsed:
        for order, entry in enumerate(parsed, 1):
            entry_valid = isinstance(entry, dict)
            item = entry if entry_valid else {}
            a = item.get('author') or {}
            a = a if isinstance(a, dict) else {}
            raw_id = a.get('id')
            aid = oa_short(raw_id, 'A')
            missing = blank(raw_id)
            invalid = not missing and aid is None
            missing_count += int(missing)
            invalid_count += int(invalid)
            valid_count += int(aid is not None)
            author_rows.append({**base,
                'authorship_record_id': p['cohort_record_id'] + f':author:{order}',
                'author_order_in_source': order, 'is_placeholder': False,
                'authorship_parse_status': status, 'authorship_entry_invalid': not entry_valid,
                'author_id_raw': raw_id, 'author_id': aid, 'author_id_missing': missing,
                'author_id_invalid': invalid, 'author_identity_unresolved': aid is None,
                'author_name': a.get('display_name') or item.get('raw_author_name'),
                'author_position': item.get('author_position'),
                'institutions_json': json.dumps(item.get('institutions'), ensure_ascii=False),
                'raw_affiliation_strings_json': json.dumps(item.get('raw_affiliation_strings'), ensure_ascii=False),
                'authorship_entry_json': json.dumps(entry, ensure_ascii=False)})
    else:
        author_rows.append({**base,
            'authorship_record_id': p['cohort_record_id'] + ':author:unknown',
            'author_order_in_source': None, 'is_placeholder': True,
            'authorship_parse_status': status, 'authorship_entry_invalid': False,
            'author_id_raw': None, 'author_id': None, 'author_id_missing': None,
            'author_id_invalid': None, 'author_identity_unresolved': True,
            'author_name': None, 'author_position': None, 'institutions_json': None,
            'raw_affiliation_strings_json': None, 'authorship_entry_json': None})
    paper_auth_flags.append({'cohort_record_id': p['cohort_record_id'],
        'authorship_parse_status': status,
        'paper_authorships_unavailable': not bool(parsed),
        'n_structured_authorship_slots': len(parsed) if parsed is not None else None,
        'n_valid_author_id_slots': valid_count,
        'n_missing_author_id_slots': missing_count if parsed is not None else None,
        'n_invalid_author_id_slots': invalid_count if parsed is not None else None,
        'paper_has_missing_author_ids': missing_count > 0,
        'paper_has_invalid_author_ids': invalid_count > 0,
        'paper_author_data_incomplete': not bool(parsed) or missing_count > 0 or invalid_count > 0 or any(not isinstance(x, dict) for x in (parsed or []))})

authors = pd.DataFrame(author_rows, columns=AUTHOR_COLUMNS)
authors['author_order_in_source'] = authors.author_order_in_source.astype('Int64')
authors['duplicate_author_id_within_record'] = authors.author_id.notna() & authors.duplicated(['cohort_record_id', 'author_id'], keep=False)
papers = papers.merge(pd.DataFrame(paper_auth_flags), on='cohort_record_id', validate='one_to_one')
for c in ['n_structured_authorship_slots', 'n_missing_author_id_slots', 'n_invalid_author_id_slots']:
    papers[c] = papers[c].astype('Int64')
assert authors.authorship_record_id.is_unique
assert set(papers.cohort_record_id) == set(authors.cohort_record_id)
assert authors.loc[~authors.is_placeholder, 'author_order_in_source'].notna().all()
assert len(authors.loc[~authors.is_placeholder]) == int(papers.n_structured_authorship_slots.sum())
display(authors[['cohort_record_id', 'author_order_in_source', 'author_name', 'author_id',
                 'author_identity_unresolved', 'is_placeholder']].head(10))

,cohort_record_id,author_order_in_source,author_name,author_id,author_identity_unresolved,is_placeholder
0,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,1,Chenjun Xiao,A5026521600,False,False
1,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2,Jincheng Mei,A5014823249,False,False
2,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,3,Martin Müller,A5010677076,False,False
3,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,1,John Hale,A5054978902,False,False
4,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2,Chris Dyer,A5111222692,False,False
5,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,3,Adhiguna Kuncoro,A5069646529,False,False
6,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,4,Jonathan Brennan,A5039799789,False,False
7,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,1,Sudha Rao,A5001484153,False,False
8,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2,Hal Daumé,A5019928111,False,False
9,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,1,Alexandre Cremers,A5040699318,False,False


## 3. Review cohort counts, exceptions and duplicate flags

Missing/invalid author IDs count **observed slots**, not guessed identities. Papers with unavailable lists are reported separately because their number of authors is unknown. A flagged paper stays in the cohort. Do not treat a parse error as a zero-author paper.

In [4]:
summary = {
    'source_rows': len(source), 'cohort_award_records': len(papers),
    'excluded_source_rows': len(excluded), 'conferences': papers.loc[~papers.paper_missing_conference, 'conference_key'].nunique(),
    'year_min': int(papers.award_year.min()), 'year_max': int(papers.award_year.max()),
    'records_with_recorded_openalex_match': int(papers.paper_has_recorded_openalex_match.sum()),
    'records_missing_openalex_id': int(papers.paper_missing_openalex_id.sum()),
    'records_invalid_openalex_id': int(papers.paper_invalid_openalex_id.sum()),
    'structured_authorship_slots': int((~authors.is_placeholder).sum()),
    'retained_slots_without_valid_author_id': int((~authors.is_placeholder & authors.author_identity_unresolved).sum()),
    'unique_valid_author_ids': authors.author_id.nunique(),
    'papers_with_missing_author_id_slots': int(papers.paper_has_missing_author_ids.sum()),
    'papers_with_invalid_author_id_slots': int(papers.paper_has_invalid_author_ids.sum()),
    'papers_with_unavailable_authorships': int(papers.paper_authorships_unavailable.sum()),
    'placeholder_rows_not_known_authors': int(authors.is_placeholder.sum()),
    'authorship_rows_including_placeholders': len(authors),
    'records_flagged_duplicate_work_id': int(papers.duplicate_openalex_work_id.sum()),
    'records_flagged_duplicate_title_year_venue': int(papers.duplicate_conference_year_title.sum())
}
summary_df = pd.DataFrame(summary.items(), columns=['measure', 'value'])
display(summary_df)
venue_year = papers.groupby(['conference', 'award_year'], dropna=False).agg(
    award_records=('cohort_record_id', 'size'),
    with_recorded_openalex_match=('paper_has_recorded_openalex_match', 'sum'),
    incomplete_author_data=('paper_author_data_incomplete', 'sum')).reset_index()
display(venue_year)

paper_flags = ['paper_missing_openalex_id', 'paper_invalid_openalex_id', 'paper_missing_conference',
               'paper_missing_title', 'duplicate_conference_year_title', 'duplicate_openalex_work_id',
               'duplicate_exact_source_record', 'paper_author_data_incomplete']
paper_issues = papers.loc[papers[paper_flags].any(axis=1)].copy()
author_issues = authors.loc[authors.author_identity_unresolved | authors.authorship_entry_invalid | authors.duplicate_author_id_within_record].copy()
display(paper_issues[['cohort_record_id', 'conference', 'award_year', 'paper_title', *paper_flags]])
display(author_issues[['cohort_record_id', 'author_order_in_source', 'author_name',
                       'author_identity_unresolved', 'is_placeholder', 'authorship_parse_status']].head(30))
display(excluded)

COUNTS_CHANGED = len(papers) != HISTORICAL_PAPER_ROWS or int(papers.paper_missing_openalex_id.sum()) != HISTORICAL_NO_OPENALEX_ID
print('Historical paper-count comparison:', 'CHANGED: review before writing' if COUNTS_CHANGED else 'agrees (925 records, 22 missing work IDs)')
print('Counts above are observed at this run. Duplicate/missing records have NOT been removed.')

,measure,value
0,source_rows,925
1,cohort_award_records,925
2,excluded_source_rows,0
3,conferences,31
4,year_min,2000
5,year_max,2018
6,records_with_recorded_openalex_match,903
7,records_missing_openalex_id,22
8,records_invalid_openalex_id,0
9,structured_authorship_slots,3202


,conference,award_year,award_records,with_recorded_openalex_match,incomplete_author_data
0,AAAI,2000,4,4,0
1,AAAI,2002,1,1,0
2,AAAI,2004,1,1,1
3,AAAI,2005,1,1,0
4,AAAI,2006,2,2,0
...,...,...,...,...,...
448,WWW,2013,1,1,0
449,WWW,2014,1,1,0
450,WWW,2015,1,1,0
451,WWW,2016,1,1,0


,cohort_record_id,conference,award_year,paper_title,paper_missing_openalex_id,paper_invalid_openalex_id,paper_missing_conference,paper_missing_title,duplicate_conference_year_title,duplicate_openalex_work_id,duplicate_exact_source_record,paper_author_data_incomplete
33,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,FOCS,2018,Approximating Edit Distance Within Constant Fa...,False,False,False,False,False,False,False,True
37,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,FSE,2018,Do Android Taint Analysis Tools Keep Their Pro...,False,False,False,False,False,False,False,True
39,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,FSE,2018,Oreo: Detection of Clones in the Twilight Zone,False,False,False,False,False,False,False,True
71,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,PLDI,2018,The Semantics of Transactions and Weak Memory ...,False,False,False,False,False,False,False,True
81,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,STOC,2018,A Constant-Factor Approximation Algorithm for ...,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
920,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,JCDL,2015,Result List Actions in Fiction Search,False,False,False,False,False,False,False,True
921,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,JCDL,2016,Comparing Published Scientific Journal Article...,False,False,False,False,False,False,False,True
922,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,JCDL,2017,Quill: A Framework for Constructing Negotiated...,False,False,False,False,False,False,False,True
923,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,JCDL,2018,Impact of Crowdsourcing OCR Improvements on Re...,False,False,False,False,False,False,False,True


,cohort_record_id,author_order_in_source,author_name,author_identity_unresolved,is_placeholder,authorship_parse_status
127,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,1,Diptarka Chakraborty,True,False,parsed
128,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2,Debarati Das,True,False,parsed
129,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,3,Elazar Goldenberg,True,False,parsed
130,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,4,Michal Koucký,True,False,parsed
131,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,5,Michael Saks,True,False,parsed
142,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,1,Felix Pauck,True,False,parsed
143,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2,Eric Bodden,True,False,parsed
144,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,3,Heike Wehrheim,True,False,parsed
154,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,1,Vaibhav Saini,True,False,parsed
155,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2,Farima Farmahinifarahani,True,False,parsed


,_source_row_number,year,conference,paper_title,paper_url,authors,openalex_id,doi,oa_title,authorships,match_route,exclusion_reason


Historical paper-count comparison: agrees (925 records, 22 missing work IDs)
Counts above are observed at this run. Duplicate/missing records have NOT been removed.


## 4. Freeze to a new directory and verify

Exports preserve all source records in scope, all parsed authorship slots, and placeholders for unknown lists. The original source bytes are archived. A manifest and data dictionary record provenance and definitions. No profile lookups, seniority decisions, author deduplication, citation calculations or control data are included.

If the destination exists, an identical source/settings freeze is **verified and reused**; it is not rewritten. A changed source/settings or damaged snapshot causes an error. For a deliberate new cohort, change `VERSION` and document why. If two source copies differ, inspect them before setting `ACK_SOURCE_COPY_DIFFERENCE = True`.

In [5]:
DEFINITIONS = {
    'cohort_record_id': 'Snapshot-specific source-row key; full source SHA-256 plus 1-based data row number.',
    'source_row_number': '1-based data record ordinal in the source CSV, not a physical text-line number.',
    'source_*': 'Original source values preserved as strings. source_authorships and source_authors preserve unstructured input.',
    'award_year': 'Integer source year, restricted to 2000–2018 inclusive.',
    'paper_has_recorded_openalex_match': 'Valid-looking recorded W ID. No API verification or match-confidence validation performed.',
    'paper_missing_openalex_id': 'Source work ID is blank or a null-like token.',
    'paper_invalid_openalex_id': 'Source work ID is nonblank but fails the accepted W-ID format.',
    'author_order_in_source': '1-based index in the source authorship list, unverified against the PDF.',
    'is_placeholder': 'Synthetic row representing unavailable structured author data; not one known author.',
    'author_id_missing': 'Missing ID within an observed author slot; blank for placeholders.',
    'author_id_invalid': 'Nonblank but invalid A-ID format within a slot; blank for placeholders.',
    'author_identity_unresolved': 'No valid A ID; true for missing/invalid slots and placeholders.',
    'paper_authorships_unavailable': 'Missing, empty, malformed or non-list structured authorships; see parse status.',
    'n_structured_authorship_slots': 'Parsed list length; blank when list cannot be parsed, zero for a valid empty list.',
    'paper_author_data_incomplete': 'Any unresolved/malformed slot or unavailable structured authorships.',
    'institutions_json': 'Institution data on this paper authorship, if supplied. Not independently verified award-date employment.',
    'duplicate_*': 'Review flags; flagged records remain in the cohort.',
    'source_exclusions.csv': 'All source rows excluded only because the year is invalid or outside 2000–2018.',
    'unit_of_analysis': 'Award-source record and author-source-record slot. Step 2 will choose the author reference event.'
}

TRANSFORM_VERSION = 'freeze_award_cohort_1.0'
SETTINGS = {'year_start': YEAR_START, 'year_end': YEAR_END, 'source_path': SOURCE_REL,
            'transform_version': TRANSFORM_VERSION, 'deduplication': 'none; flags only',
            'author_policy': 'retain missing IDs and unknown-list placeholders'}

def verify_snapshot(folder, expected_source_sha, expected_settings):
    manifest = json.loads((folder / 'manifest.json').read_text(encoding='utf-8'))
    if manifest['source_sha256'] != expected_source_sha or manifest['settings'] != expected_settings:
        raise RuntimeError('Existing freeze has different source/settings. Do not overwrite; review and version deliberately.')
    for name, expected in manifest['output_sha256'].items():
        file = folder / name
        if not file.is_file() or sha256_bytes(file.read_bytes()) != expected:
            raise RuntimeError(f'Frozen output failed integrity check: {name}')
    return manifest

if WRITE_FREEZE:
    if SOURCE_COPY_DIFFER and not ACK_SOURCE_COPY_DIFFERENCE:
        raise RuntimeError('Source copies differ. Inspect before acknowledging the primary source choice.')
    if COUNTS_CHANGED and not ACK_CHANGED_PAPER_COUNTS:
        raise RuntimeError('Paper counts differ from the historical summary. Review before setting ACK_CHANGED_PAPER_COUNTS=True.')
    if TARGET.exists():
        manifest = verify_snapshot(TARGET, SOURCE_SHA256, SETTINGS)
        print('Verified existing freeze; nothing rewritten:', TARGET)
    else:
        TARGET.parent.mkdir(parents=True, exist_ok=True)
        staging = Path(tempfile.mkdtemp(prefix='.' + VERSION + '_', dir=TARGET.parent))
        try:
            tables = {'award_cohort_papers.csv': papers,
                      'award_cohort_authorships.csv': authors,
                      'paper_issues.csv': paper_issues, 'author_issues.csv': author_issues,
                      'source_exclusions.csv': excluded, 'cohort_summary.csv': summary_df,
                      'cohort_conference_year_counts.csv': venue_year}
            for name, frame in tables.items():
                frame.to_csv(staging / name, index=False)
            (staging / 'source_awards_snapshot.csv').write_bytes(source_bytes)
            if copy_bytes is not None:
                (staging / 'journal_copy_snapshot.csv').write_bytes(copy_bytes)
            (staging / 'data_dictionary.json').write_text(json.dumps(DEFINITIONS, ensure_ascii=False, indent=2), encoding='utf-8')
            (staging / 'README.md').write_text(
                '# Frozen award cohort: ' + VERSION + '\n\n'
                'Award-source records from 2000–2018, including unmatched and missing-author-ID records. '
                'No deduplication, no author reference-award selection, no new API calls.\n\n'
                'Use award_cohort_papers.csv as the paper roster and award_cohort_authorships.csv as the linked authorship roster. '
                'Join using cohort_record_id, not conference/year. Exclude is_placeholder rows only when counting known author slots. '
                'See manifest.json, data_dictionary.json and issue tables. Do not edit a frozen snapshot in place.\n', encoding='utf-8')
            readback_papers = pd.read_csv(staging / 'award_cohort_papers.csv', dtype=str, keep_default_na=False)
            readback_authors = pd.read_csv(staging / 'award_cohort_authorships.csv', dtype=str, keep_default_na=False)
            assert len(readback_papers) == len(papers)
            assert len(readback_authors) == len(authors)
            assert readback_papers.cohort_record_id.is_unique
            assert set(readback_papers.cohort_record_id) == set(readback_authors.cohort_record_id)
            output_hashes = {p.name: sha256_bytes(p.read_bytes()) for p in sorted(staging.iterdir()) if p.is_file()}
            manifest = {'cohort_version': VERSION, 'created_at_utc': datetime.now(timezone.utc).isoformat(),
                        'git_commit_at_run': git_value(['rev-parse', 'HEAD']),
                        'source_sha256': SOURCE_SHA256, 'journal_copy_path': COPY_REL,
                        'journal_copy_sha256': COPY_SHA256, 'source_copy_difference_acknowledged': ACK_SOURCE_COPY_DIFFERENCE,
                        'changed_paper_counts_acknowledged': ACK_CHANGED_PAPER_COUNTS,
                        'settings': SETTINGS, 'summary': {k: int(v) for k, v in summary.items()},
                        'output_sha256': output_hashes,
                        'scope_limitations': ['Observed source cohort, not independently complete award census.',
                            'Recorded OpenAlex IDs are not revalidated.', 'Authorship list order/affiliations are source metadata.',
                            'Unavailable lists do not imply zero authors.']}
            (staging / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
            verify_snapshot(staging, SOURCE_SHA256, SETTINGS)
            if TARGET.exists():
                raise FileExistsError('Freeze directory appeared during export; refusing to overwrite.')
            staging.rename(TARGET)
            print('Created and verified new frozen cohort:', TARGET)
        finally:
            if staging.exists():
                shutil.rmtree(staging)
else:
    print('Preview only. No frozen files written. Set WRITE_FREEZE=True to create the snapshot.')

Created and verified new frozen cohort: B:\Semester 4 UU\thesis-best-paper-trajectories\conf_paper\conf_data\frozen\award_cohort_v1


## Step 1 completion checklist

When the final cell reports **Created and verified** (or verifies an existing matching snapshot):
1. Inspect the displayed summary and exception tables.
2. Keep all unmatched award records and missing-ID authorship entries in the frozen roster.
3. Commit this executed notebook and the new versioned snapshot directory after review.
4. Use the frozen outputs for subsequent work; do not modify them to add later metrics. Produce separate downstream tables instead.

The expected historical benchmarks are 925 award records and 22 missing work IDs; the notebook computes actual values rather than hardcoding them into the results. A successful freeze preserves a reproducible starting cohort. It does **not** mean that match quality, seniority, institutions or citation histories have already been validated.